# 07 — Freeze Fresh Unified DEV Subject Universe

**Version:** `SAJU_ML_V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE_20260816`

This notebook selects the next **100 fresh development subjects before event collection and before any astrology scoring**.

Target allocation:

```text
COMPETITIVE  30
PROJECT      25
STATUS       45
TOTAL       100
```

It does not collect events, fit models, generate astrology features, or open a holdout.

The purpose is to freeze the **subject + birth record + preassigned axis universe first**, so later event collection cannot choose people because their life histories happen to be convenient.

In [1]:
from pathlib import Path
from datetime import datetime
import hashlib
import json
import re
import unicodedata
import urllib.request

import numpy as np
import pandas as pd

NOTEBOOK_VERSION = "SAJU_ML_V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE_20260816"
SEED = 20260816

BIRTH_URL = (
    "https://huggingface.co/datasets/vedastro-org/"
    "15000-Famous-People-Birth-Date-Location/raw/main/PersonList-15k.csv"
)

TARGET = {"COMPETITIVE": 30, "PROJECT": 25, "STATUS": 45}
BIRTH_YEAR_MIN = 1900
BIRTH_YEAR_MAX = 1995

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for candidate in [p] + list(p.parents):
        if (candidate / "saju_engine.py").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the Chartpalja saju repo.")

REPO_ROOT = find_repo_root()
OUT = REPO_ROOT / "research/ml/artifacts/v4_unified_dev_roster"
OUT.mkdir(parents=True, exist_ok=True)

SEED_PATH = REPO_ROOT / "research/ml_corpus/v4_unified_dev/V4_UNIFIED_DEV_PREDECLARED_SEED_UNIVERSE.csv"
CONTRACT_PATH = REPO_ROOT / "research/ml_corpus/v4_unified_dev/V4_UNIFIED_DEV_COLLECTION_CONTRACT.json"

print("repo:", REPO_ROOT)
print("output:", OUT)
print("version:", NOTEBOOK_VERSION)

repo: /Users/sangjinlee/Desktop/projects/saju
output: /Users/sangjinlee/Desktop/projects/saju/research/ml/artifacts/v4_unified_dev_roster
version: SAJU_ML_V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE_20260816


## 1. Holdout-safe prior-subject exclusion

In [2]:
def norm_name(x):
    s = unicodedata.normalize("NFKD", str(x))
    s = "".join(c for c in s if not unicodedata.combining(c))
    s = s.casefold()
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

def extract_subject_names_from_corpus(path):
    if not path.exists():
        return []
    with open(path, encoding="utf-8") as f:
        obj = json.load(f)
    return [
        s.get("name")
        for s in obj.get("subjects", [])
        if s.get("name")
    ]

# Only development/consumed corpora are loaded.
# Do not fall back to any combined V2 corpus.
prior_paths = {
    "V1": REPO_ROOT / "research/ml_corpus/v1/SAJU_ML_CORPUS_V1.json",
    "V2_NEW_DEV": REPO_ROOT / "research/ml_corpus/v2/SAJU_ML_CORPUS_V2_NEW_DEV.json",
    "NEW_DEV_2_CONSUMED": REPO_ROOT / "research/ml_corpus/new_dev_2/SAJU_ML_NEW_DEV_2_CORPUS.json",
    "V4_TARGET_EXPANSION": REPO_ROOT / "research/ml_corpus/v4_target_expansion/V4_TARGET_EXPANSION_CORPUS.json",
}

prior_names = []
for label, path in prior_paths.items():
    names = extract_subject_names_from_corpus(path)
    print(label, len(names), "subjects loaded for exclusion")
    prior_names.extend(names)

prior_norm = {norm_name(x) for x in prior_names}

assert "NEW_CONFIRM" not in " ".join(str(p) for p in prior_paths.values()).upper()
print("prior unique normalized names:", len(prior_norm))
print("sealed-set paths: NOT DEFINED")

V1 94 subjects loaded for exclusion
V2_NEW_DEV 70 subjects loaded for exclusion
NEW_DEV_2_CONSUMED 100 subjects loaded for exclusion
V4_TARGET_EXPANSION 169 subjects loaded for exclusion
prior unique normalized names: 232
sealed-set paths: NOT DEFINED


## 2. Load the predeclared seed universe and Rodden-AA birth source

In [3]:
seed_df = pd.read_csv(SEED_PATH)
with open(CONTRACT_PATH, encoding="utf-8") as f:
    contract = json.load(f)

assert contract["status"] == "SUBJECT_UNIVERSE_PREDECLARED_BEFORE_EVENT_COLLECTION"
assert seed_df.axis.isin(TARGET).all()

cache_path = OUT / "PersonList-15k.csv"

if not cache_path.exists():
    print("Downloading public Rodden-AA birth dataset...")
    try:
        urllib.request.urlretrieve(BIRTH_URL, cache_path)
    except Exception as e:
        raise RuntimeError(
            "Could not download VedAstro birth dataset. "
            "Download PersonList-15k.csv manually from the URL in the collection contract "
            "and place it at: %s\nOriginal error: %r" % (cache_path, e)
        )

birth = pd.read_csv(cache_path)
required_cols = {"RowKey","BirthTime","Gender","Name","Notes"}
assert required_cols.issubset(birth.columns)

birth = birth[birth["Notes"].astype(str).str.contains("AA", regex=False)].copy()
birth["norm_name"] = birth["Name"].map(norm_name)
print("AA birth rows:", len(birth))

AA birth rows: 15790


## 3. Parse birth records and resolve seed names

In [4]:
def parse_birth_blob(blob):
    obj = json.loads(str(blob))
    std = obj["StdTime"]
    loc = obj["Location"]

    # Example: 18:50 12/03/1968 -08:00
    m = re.match(
        r"^(\d{1,2}):(\d{2})\s+(\d{2})/(\d{2})/(\d{4})\s+([+-]\d{2}:\d{2})$",
        std.strip()
    )
    if not m:
        raise ValueError("Unexpected StdTime: %r" % std)

    hh, mm, dd, mo, yyyy, offset = m.groups()
    return {
        "birth_date": "%04d-%02d-%02d" % (int(yyyy), int(mo), int(dd)),
        "birth_time": "%02d:%02d" % (int(hh), int(mm)),
        "utc_offset": offset,
        "birth_year": int(yyyy),
        "birth_place": loc["Name"],
        "longitude": float(loc["Longitude"]),
        "latitude": float(loc["Latitude"]),
    }

parsed_rows = []
for _, r in birth.iterrows():
    try:
        p = parse_birth_blob(r["BirthTime"])
    except Exception:
        continue
    parsed_rows.append({
        "source_row_key": r["RowKey"],
        "source_name": r["Name"],
        "gender": str(r["Gender"]).lower(),
        "norm_name": r["norm_name"],
        **p,
    })

birth_parsed = pd.DataFrame(parsed_rows)

seed_df["norm_name"] = seed_df["name"].map(norm_name)

resolved = seed_df.merge(
    birth_parsed,
    on="norm_name",
    how="left",
    suffixes=("_seed","_birth"),
)

resolved["prior_used"] = resolved["norm_name"].isin(prior_norm)
resolved["birth_in_range"] = resolved["birth_year"].between(
    BIRTH_YEAR_MIN, BIRTH_YEAR_MAX, inclusive="both"
)
resolved["eligible"] = (
    resolved["source_row_key"].notna()
    & ~resolved["prior_used"]
    & resolved["birth_in_range"]
)

resolution_summary = (
    resolved.groupby("axis")
    .agg(
        seeds=("name","size"),
        resolved_aa=("source_row_key", lambda x: x.notna().sum()),
        prior_used=("prior_used","sum"),
        eligible=("eligible","sum"),
    )
    .reset_index()
)

display(resolution_summary)

for axis, n in TARGET.items():
    available = int(resolution_summary.loc[resolution_summary.axis==axis, "eligible"].iloc[0])
    if available < n:
        raise RuntimeError(
            "%s has only %d eligible seed subjects; need %d. "
            "Expand the predeclared seed universe BEFORE event collection."
            % (axis, available, n)
        )

,axis,seeds,resolved_aa,prior_used,eligible
0,COMPETITIVE,116,35,6,30
1,PROJECT,107,48,22,29
2,STATUS,345,95,7,84


## 4. Deterministic selection with a gender guardrail

In [5]:
def deterministic_key(name, axis):
    raw = ("%s|%s|%s" % (SEED, axis, norm_name(name))).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()

eligible = resolved[resolved["eligible"]].copy()
eligible["det_key"] = [
    deterministic_key(n, a)
    for n, a in zip(eligible["name"], eligible["axis"])
]

selected_parts = []

for axis, target_n in TARGET.items():
    pool = eligible[eligible.axis == axis].copy().sort_values("det_key")

    # Guardrail, not post-hoc outcome balancing:
    # reserve at least 20% female if the predeclared eligible pool permits it.
    female_target = int(np.ceil(target_n * 0.20))
    women = pool[pool.gender.str.startswith("female")].head(female_target)
    chosen = women.copy()

    remaining = pool[~pool.norm_name.isin(chosen.norm_name)].copy()
    chosen = pd.concat([chosen, remaining.head(target_n - len(chosen))])

    if len(chosen) != target_n:
        raise RuntimeError("%s selection failed: %d / %d" % (axis, len(chosen), target_n))

    selected_parts.append(chosen)

roster = pd.concat(selected_parts, ignore_index=True)

assert len(roster) == 100
assert roster.norm_name.nunique() == 100
assert not roster.prior_used.any()

roster["subject_id"] = [
    "V4UD_%03d" % (i+1)
    for i in range(len(roster))
]
roster["preassigned_axis"] = roster["axis"]
roster["rodden_rating"] = "AA"
roster["birth_source"] = "VedAstro 15K / Rodden AA"
roster["birth_source_url"] = BIRTH_URL
roster["event_collection_started"] = False
roster["astrology_scored"] = False

cols = [
    "subject_id","name","source_name","gender","preassigned_axis",
    "birth_date","birth_time","utc_offset","birth_place","latitude","longitude",
    "rodden_rating","source_row_key","birth_source","birth_source_url",
    "event_collection_started","astrology_scored","det_key",
]
roster = roster[cols].sort_values(["preassigned_axis","det_key"]).reset_index(drop=True)

summary = (
    roster.groupby("preassigned_axis")
    .agg(
        n_subjects=("subject_id","size"),
        n_female=("gender", lambda x: x.str.startswith("female").sum()),
        birth_year_min=("birth_date", lambda x: pd.to_datetime(x).dt.year.min()),
        birth_year_max=("birth_date", lambda x: pd.to_datetime(x).dt.year.max()),
    )
    .reset_index()
)
summary["female_share"] = summary["n_female"] / summary["n_subjects"]

display(summary)

,preassigned_axis,n_subjects,n_female,birth_year_min,birth_year_max,female_share
0,COMPETITIVE,30,6,1918,1987,0.200000
1,PROJECT,25,5,1916,1972,0.200000
2,STATUS,45,4,1907,1975,0.088889


## 5. Freeze roster before any event search

In [6]:
roster_path = OUT / "V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv"
roster.to_csv(roster_path, index=False)

roster_hash = hashlib.sha256(roster_path.read_bytes()).hexdigest()

freeze = {
    "version": "V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE_V1",
    "notebook_version": NOTEBOOK_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "status": "V4_UNIFIED_DEV_SUBJECT_UNIVERSE_FROZEN",
    "n_subjects": int(len(roster)),
    "axis_counts": roster.preassigned_axis.value_counts().to_dict(),
    "roster_sha256": roster_hash,
    "selection_seed": SEED,
    "birth_source_url": BIRTH_URL,
    "prior_development_names_excluded": int(len(prior_norm)),
    "astrology_scored": False,
    "event_collection_started": False,
    "sealed_holdouts_loaded": False,
    "next_rule": (
        "Do not change subject membership after event collection begins. "
        "Collect all source-verifiable opposite-polarity Career events on each subject's "
        "preassigned axis, then apply the minimum-absolute-year-gap pair rule."
    ),
}

with open(OUT / "V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE.json","w",encoding="utf-8") as f:
    json.dump(freeze, f, ensure_ascii=False, indent=2)

print(json.dumps(freeze, ensure_ascii=False, indent=2))

{
  "version": "V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE_V1",
  "notebook_version": "SAJU_ML_V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE_20260816",
  "created_at": "2026-08-16T20:00:30",
  "status": "V4_UNIFIED_DEV_SUBJECT_UNIVERSE_FROZEN",
  "n_subjects": 100,
  "axis_counts": {
    "STATUS": 45,
    "COMPETITIVE": 30,
    "PROJECT": 25
  },
  "roster_sha256": "8fc11f61c5102f3c2caa9890ec82819e164178ac0a96f4d721f4e13767fdc9a7",
  "selection_seed": 20260816,
  "birth_source_url": "https://huggingface.co/datasets/vedastro-org/15000-Famous-People-Birth-Date-Location/raw/main/PersonList-15k.csv",
  "prior_development_names_excluded": 232,
  "astrology_scored": false,
  "event_collection_started": false,
  "sealed_holdouts_loaded": false,
  "next_rule": "Do not change subject membership after event collection begins. Collect all source-verifiable opposite-polarity Career events on each subject's preassigned axis, then apply the minimum-absolute-year-gap pair rule."
}


## 6. Create event-collection intake sheet

In [7]:
# 3 positive + 3 negative candidate-event slots per subject.
# Add rows if a subject has more verified events; do not delete inconvenient events.
intake_rows = []
for _, s in roster.iterrows():
    for polarity in ["positive","negative"]:
        for slot in range(1,4):
            intake_rows.append({
                "subject_id": s.subject_id,
                "name": s["name"],
                "preassigned_axis": s.preassigned_axis,
                "candidate_polarity": polarity,
                "slot": slot,
                "event_year": "",
                "event_date_if_known": "",
                "event_type": "",
                "event_description": "",
                "primary_source_url": "",
                "secondary_source_url": "",
                "source_quality": "",
                "taxonomy_confirmed": False,
                "polarity_confirmed": False,
                "source_confirmed": False,
                "exclude_reason_if_invalid": "",
                "notes": "",
            })

intake = pd.DataFrame(intake_rows)
intake.to_csv(OUT / "V4_UNIFIED_DEV_EVENT_CANDIDATE_INTAKE.csv", index=False)

print("event intake rows:", len(intake))
print("Next step: collect events WITHOUT astrology scores.")

event intake rows: 600
Next step: collect events WITHOUT astrology scores.


## What to send back

After **Kernel Restart → Run All**, send:

```text
research/ml/artifacts/v4_unified_dev_roster/

V4_UNIFIED_DEV_SUBJECT_ROSTER_100.csv
V4_UNIFIED_DEV_SUBJECT_ROSTER_FREEZE.json
V4_UNIFIED_DEV_EVENT_CANDIDATE_INTAKE.csv
```

Expected freeze status:

```text
V4_UNIFIED_DEV_SUBJECT_UNIVERSE_FROZEN
```

Do not generate astrology features yet.

Once the roster is frozen, event/source collection can begin without subject-selection leakage.